# ![icon](./images/uva-icon-57x57.png) WEEK 06 Code Cleanup and OOP with AI Assist


| Working Efficiently With Software |
| :--: |
| Data Engineering |
| School of Data Science |
| University of Virginia |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda


**This week and next are one extended lecture and two labs**
* I'll release the labs on Wednesday so you have them for the weekend and through next week.
* **Lab 06a:  Cleanup**
* **Lab 06b:  OOP our pipeline with AI assist**


## IMPORTANT: Let's do this now please
* Lets add `karolinaranjo` to your github repo as a collaborator
* She will be helping me provide feedback on the next two labs


## I'm upping the quality control on the Pull Requests
So far I've not been taking off points for some smaller details.
* Notably -> missing lab links on PR descriptons.
    - These seemingly small rules will now be a source of bounce back, or points lost.
    - Not because I'm a stickler, but because in many work environments this is **automated**.   Your PR won't be seen unless you hit the right automation points

Now that you have some time working in the command line, in git, we'll use the cleanup lab to start placing other practices, habits that should just be natural in your process.


## Today's lecture
* Overview of the cleanup lab 06a
* OOP intuition
* Describe script we'll be using for the OOP lab and a description of the OOP lab 06b
* OOP and Design Concepts

## Regarding the "cleanup" lab
I'm am completing a list of general items that are the most common gotchas.

For example
* keeping *,pyc files out of your repo, and maintaining a solid gitignore.
* How to handle failing jobs in makefiles
* Refining our cy.yml to get the most out of it.

**ETC**

This lab may extend and have more than one branch.  In fact it may have many branches as you fix things one by one.

Git logging will be very important here, as in your lab 06b

# ![icon](./images/uva-icon-57x57.png) The concepts are universal

### Metaphor: The Pipeline as LEGO Bricks

> "Write programs that do one thing and do it well. Write programs to work together. Write programs to handle text streams, because that is a universal interface." — Doug McIlroy (Inventor of the UNIX Pipe)

---

### Core Invariant Infrastructure (The Terminal Stream)

```bash
# A production-grade UNIX data pipeline
cat video_ids.txt | grep -v "^#" | sort -u | ./extract.py --source youtube | ./enrich.py > staged.jsonl

```

### 🔍 Architectural Code Breakdown

* **`cat video_ids.txt`** * *Responsibility:* Streams raw text from disk to `stdout`.
* *The Interface Law:* **`cat` does not sort.** If `cat` secretly tried to reorder or alter data, it would break its structural contract, making the system unpredictable.


* **`grep -v "^#"`**
* *Responsibility:* Filters out comment lines. It does not know or care where the stream came from (`cat`, a file, or a network socket).


* **`sort -u`**
* *Responsibility:* Deduplicates and reorders. It doesn't know what a "Video ID" is; it only understands alpha-numeric strings.


* **`./extract.py | ./enrich.py`**
* *Responsibility:* Custom business nodes. Because they use `stdin` and `stdout`, they plug seamlessly into ancient UNIX utilities built 50 years ago.



---

### Key Architecture Takeaways for Software Design

* **The Universal Interface (Duck Typing):** Every command expects a text stream (`stdin`) and emits a text stream (`stdout`). As long as it "quacks" like a stream, the pipeline components link together perfectly.
* **The Open-Closed Principle (OCP):** If you want to add a logging step tomorrow, you don't rewrite `cat`, `sort`, or your scripts. You simply extend the chain:
```bash
cat ids.txt | sort -u | tee -a pipeline.log | ./extract.py

```


* **Encapsulation & Isolation:** What happens inside `extract.py` (API tokens, proxy matrices, network retries) is completely hidden from `enrich.py`. Components are entirely isolated black boxes that reason cleanly, test easily, and swap instantly.

# OOP Design Concepts

### **The Open-Closed Principle (OCP)**

* **The Rule:** Code should be open for extension, but closed for modification.
* **The Pipeline Application:** Add a brand-new ingestion source or LLM vendor by writing a new class—never by breaking open your stable, core orchestrator loop.
* **The Goal:** Scale up functionality without introducing regressions.

### **The Dependency Inversion Principle (DIP)**

* **The Rule:** High-level orchestration should never depend directly on low-level infrastructure details; both must rely on abstract contracts (interfaces).
* **The Pipeline Application:** Force your main loop to talk to a generic wrapper interface rather than a specific vendor SDK (Gemini/Claude).
* **The Goal:** Insulate your system's core plumbing from third-party API version shifts.

---

# Python Engine Mechanics

### **Duck Typing**

* **The Rule:** An object's suitability is determined by its behavior and methods, not its explicit inheritance class ("If it walks and quacks like a duck...").
* **The Pipeline Application:** If a new object has a working `.fetch_raw_string()` method, your invariant stream loop will run it automatically.
* **The Goal:** Treat entirely different networking topologies (API scrapers vs. XML parsers) as completely interchangeable.

### **Interface over Implementation**

* **The Rule:** Program to what an object *can do*, not *how* it currently executes the task.
* **The Pipeline Application:** Lock down method contracts so you can seamlessly swap data-cleaning or extraction algorithms at runtime.
* **The Goal:** Turn hardcoded, fragile scripts into modular, pluggable LEGO bricks.

### **Encapsulation (Data Hiding)**

* **The Rule:** Hide internal boilerplate chaos inside a class capsule; expose only a clean, simple public method.
* **The Pipeline Application:** Your main engine shouldn't see proxy rotation rules or XML node parsing tags; it simply calls `.process()` and trusts the object to handle its own mess.
* **The Goal:** Minimize cognitive load and keep distinct engineering problems in distinct boxes.

# ![icon](./images/uva-icon-57x57.png) OOP Pipeline

In [14]:
import re
from abc import ABC, abstractmethod

# =====================================================================
# THE ANTI-PATTERN: PROCEDURAL STRING TRANSFORMATION (TIGHT COUPLING)
# =====================================================================
# Why this fails enterprise scaling: 
# Every single time your business requirements introduce a new type of data
# cleanup rule, you are forced to break open this function and add another
# conditional block. This violates the Open-Closed Principle (OCP).
# =====================================================================

def procedural_transformer(raw_text: str, mode: str) -> str:
    """A legacy procedural block that handles multiple cleaning styles."""
    if mode == "standard":
        # Strip outer whitespace and force lowercase
        return raw_text.strip().lower()
        
    elif mode == "strip_punctuation":
        # Use regex to strip out non-alphanumeric characters
        return re.sub(r'[^\w\s]', '', raw_text)
        
    # REFACTOR BLOCKPOINT: 
    # If we need an "anonymize" or "uppercase" mode, we must surgically 
    # edit this core pipeline loop, increasing the risk of code regression.
    else:
        raise ValueError(f"Unknown transformation mode: {mode}")


# =====================================================================
# THE SOLUTION: OBJECT-ORIENTED PIPELINES (DEPENDENCY INVERSION)
# =====================================================================
# 1. Fluent Python Ch 13: We use Python's `abc` module to draw a strict,
#    enforceable structural contract (an interface).
# 2. Architecture Patterns Ch 1 & 2: We isolate the variant business rules
#    from the invariant orchestration plumbing.
# =====================================================================

# 1. THE CONTRACT (Abstraction)
class TextStrategy(ABC):
    """
    The Abstract Base Class interface blueprint.
    This dictates WHAT a text strategy must do, without caring HOW it does it.
    """
    @abstractmethod
    def process(self, text: str) -> str:
        """Enforces that any child strategy must implement this method signature."""
        pass


class SomeOther():
    def process(self, text: str) -> str:
        return 'abc'

# 2. THE CONCRETE WORKERS (Encapsulation)
class StandardCleaner(TextStrategy):
    """Capsules the logic for lightweight data normalization."""
    def process(self, text: str) -> str:
        return text.strip().lower()

class PunctuationStripper(TextStrategy):
    """Capsules the logic for regex-driven structural cleaning."""
    def process(self, text: str) -> str:
        """
        INPUT text   str ...dec

        RETURN:
            string
        """
        return re.sub(r'[^\w\s]', '', text)

class UppercaseStrategy(TextStrategy):
    def process(self, text: str) -> str:
        return text.upper()


# 4. THE CONTEXT / PLUMBING (Polymorphism & Dependency Injection)
class DataTransformationNode:
    """
    The Invariant Pipeline Engine.
    This class handles the core loop infrastructure. Notice that it contains
    zero string manipulation code, zero URLs, and zero if/else condition blocks.
    It relies entirely on Polymorphism to execute the correct algorithm.
    """
    def __init__(self, transformation_strategy: TextStrategy):
        # Dependency Injection: We inject the contract interface, not a hardcoded worker
        self.strategy = transformation_strategy

    def execute_pipeline(self, sample_data: list) -> list:
        cleaned_dataset = []
        for row in sample_data:
            # Polymorphic Link: Python dynamically tracks down the correct .process() 
            # execution path based on whatever object was injected at runtime.
            transformed_row = self.strategy.process(row)
            cleaned_dataset.append(transformed_row)
        return cleaned_dataset


# =====================================================================
# INTERACTIVE RUNTIME VALIDATION BLOCK
# =====================================================================
if __name__ == "__main__":
    # Sample dirty pipeline stream input
    dirty_transcripts = [
        "   HELLO WORLD!!! This is a data stream.   ",
        "Pipeline tools: Polars, Docker, and dbt.",
        "   CRITICAL ERROR caught on line 42...   "
    ]

    print("--- 1. EXECUTING PROCEDURAL ANTI-PATTERN ---")
    try:
        standard_out = [procedural_transformer(line, "standard") for line in dirty_transcripts]
        print(f"Standard Out: {standard_out}\n")
    except Exception as e:
        print(f"Procedural Error: {e}")



    
    print("--- 2. EXECUTING OBJECT-ORIENTED STRATEGY PATTERN ---")
    # Step A: Instantiate our chosen worker capsule
    choice = 'upper'
    chosen_worker = {
        "punct": PunctuationStripper(),
        "stand": StandardCleaner(),
        "upper": UppercaseStrategy(),
    }.get(choice, None)
    assert choice, f"Invalid selection {choice}"
    
    # Step B: Inject that worker into our generic pipeline node execution context
    pipeline_node = DataTransformationNode(chosen_worker)
    
    # Step C: Stream data through the engine flawlessly
    processed_output = pipeline_node.execute_pipeline(dirty_transcripts)
    print(f"OO-Strategy Out: {processed_output}")

--- 1. EXECUTING PROCEDURAL ANTI-PATTERN ---
Standard Out: ['hello world!!! this is a data stream.', 'pipeline tools: polars, docker, and dbt.', 'critical error caught on line 42...']

--- 2. EXECUTING OBJECT-ORIENTED STRATEGY PATTERN ---
OO-Strategy Out: ['   HELLO WORLD!!! THIS IS A DATA STREAM.   ', 'PIPELINE TOOLS: POLARS, DOCKER, AND DBT.', '   CRITICAL ERROR CAUGHT ON LINE 42...   ']


# ![icon](./images/uva-icon-57x57.png) KEY TAKEAWAY

**We started with Bash pipelines to show you what absolute isolation looks like in the wild.**

**The ultimate goal of software engineering is to design code that functions exactly like those interchangeable LEGO bricks—even when you step away from the command line.**

**By applying OOP contracts and Design Patterns, you aren't just writing code; you are building an internal infrastructure grid where components test easily, swap instantly, and scale without friction.**

# ![icon](./images/uva-icon-57x57.png) Overview of LAB06b
In this lab you will be.

* Use the working model for how the `extract_transcripts.py` module is adapted for the Stragety Pattern to apply it to the `enrich_transcripts.py`.
* You will use a prompt to use in Cluade.ai so you can work with it in a more **Socratic** collaboration.
* Your goal will be to use claude.ai to help you reproduce the script by interaction.

## IMPORTANT
* Do not just ask it to OOP and check in the code
* Deliverable 1 will include a link to the chat or a summary of the chat (which can be prompted)
* Deliverable 2 will be a series of properly commented, atomic git commits that read clearly in github.
* Deliverable 3 will be the updated `extract_transcripts.py`

The **Goal** of the assignment, is to practice using AI not as a 'vibe code' tool, but as a collaborator/guide.

You **don't** want to give up control, understanding of the code.  But given what you do understand after this lecture, you should be able to collaborate, and learn from the AI to complete the task.

**NB:** This is the pilot program.  So if something is not clear, fuzzy, or you just get stuck, **DO NOT PROCEED:** reach out and grab me or Karoline and we will work through it in office hours, email, Teams or when you are available.

## Working file for the extract_transcripts.py OOP version
```python
#!/usr/bin/env python3
import sys
import json
import argparse
import os
from abc import ABC, abstractmethod

# =====================================================================
# 1. THE EXTRACTION CONTRACT (Interface)
# =====================================================================
class TranscriptExtractor(ABC):
    @abstractmethod
    def fetch_raw_string(self, source_id: str) -> str:
        """Must accept an identifier string and return raw, unstructured text data."""
        pass


# =====================================================================
# 2. STRATEGY A: THE PRODUCTION YOUTUBE SCRAPER (Version-Agnostic)
# =====================================================================
class YouTubeExtractor(TranscriptExtractor):
    def __init__(self, proxy_url: str = None):
        self.proxy_url = proxy_url

    def fetch_raw_string(self, source_id: str) -> str:
        from youtube_transcript_api import YouTubeTranscriptApi

        original_http = os.environ.get("HTTP_PROXY")
        original_https = os.environ.get("HTTPS_PROXY")
        
        if self.proxy_url:
            os.environ["HTTP_PROXY"] = self.proxy_url
            os.environ["HTTPS_PROXY"] = self.proxy_url

        try:
            if hasattr(YouTubeTranscriptApi, 'get_transcript'):
                proxies_dict = {"http": self.proxy_url, "https": self.proxy_url} if self.proxy_url else None
                transcript_list = YouTubeTranscriptApi.get_transcript(source_id, proxies=proxies_dict)
            else:
                client = YouTubeTranscriptApi()
                if hasattr(client, 'fetch'):
                    transcript_list = client.fetch(source_id).to_raw_data()
                else:
                    transcript_list = client.get_transcript(source_id)

            cleaned_text = " ".join([entry["text"] for entry in transcript_list])
            return cleaned_text
            
        except Exception as e:
            raise RuntimeError(f"YouTube transcript extraction failed for {source_id}: {str(e)}")
        finally:
            if self.proxy_url:
                if original_http: os.environ["HTTP_PROXY"] = original_http
                else: os.environ.pop("HTTP_PROXY", None)
                if original_https: os.environ["HTTPS_PROXY"] = original_https
                else: os.environ.pop("HTTPS_PROXY", None)


# =====================================================================
# 3. STRATEGY B: THE LIVE HTTP CLOUD ARCHIVE (Podcast RSS)
# =====================================================================
class PodcastRssExtractor(TranscriptExtractor):
    def __init__(self, feed_url: str):
        self.feed_url = feed_url

    def fetch_raw_string(self, source_id: str) -> str:
        import requests
        import xml.etree.ElementTree as ET

        try:
            response = requests.get(self.feed_url, timeout=10)
            if response.status_code != 200:
                raise RuntimeError(f"HTTP Error {response.status_code} accessing RSS feed.")
        except Exception as e:
            raise RuntimeError(f"Network transport failure connecting to feed: {str(e)}")

        try:
            root = ET.fromstring(response.content)
        except Exception as e:
            raise ValueError(f"Failed to compile valid XML tree payload: {str(e)}")

        for item in root.findall('.//item'):
            title_elem = item.find('title')
            desc_elem = item.find('description')
            
            if title_elem is not None and desc_elem is not None:
                title_text = title_elem.text
                if source_id.lower() in title_text.lower():
                    return f"[LIVE PODCAST SHOW NOTES - {title_text}]: {desc_elem.text}"

        raise ValueError(f"No podcast episode matching keyword '{source_id}' located in RSS catalog.")


# =====================================================================
# 4. THE INVARIANT PIPELINE CONTEXT (The Streaming Engine)
# =====================================================================
class ExtractionEngine:
    def __init__(self, strategy: TranscriptExtractor):
        self.strategy = strategy

    def run_stream(self):
        for line in sys.stdin:
            source_id = line.strip()
            if not source_id:
                continue

            try:
                raw_text = self.strategy.fetch_raw_string(source_id)
                
                # FIXED: Kept identical legacy key schema names to protect older tests
                payload = {
                    "video_id": source_id,
                    "raw_text": raw_text
                }
                sys.stdout.write(json.dumps(payload) + "\n")
                sys.stdout.flush()
                
            except Exception as e:
                sys.stderr.write(f"ERROR processing token [{source_id}]: {str(e)}\n")
                sys.stderr.flush()


# =====================================================================
# 5. RUNTIME ENTRYPOINT
# =====================================================================
# FIXED: Accept optional argv parameter for native test-framework compatibility
def main(argv=None):
    parser = argparse.ArgumentParser(description="Multi-Source Transcript Ingestion Node.")
    parser.add_argument(
        "--source", 
        choices=["youtube", "podcast"], 
        default="youtube",
        help="Target infrastructure ingestion path strategy (Defaults to youtube)."
    )
    parser.add_argument("--proxy", default=None, help="Optional proxy endpoint URL string.")
    
    # FIXED: Pass the argv variable into parse_args directly
    args = parser.parse_args(argv)

    if args.source == "youtube":
        selected_strategy = YouTubeExtractor(proxy_url=args.proxy)
    elif args.source == "podcast":
        selected_strategy = PodcastRssExtractor(feed_url="https://talkpython.fm/episodes/rss")

    engine = ExtractionEngine(selected_strategy)
    engine.run_stream()

if __name__ == "__main__":
    main()
```



# ![icon](./images/duck_typing.jpg)